<a href="https://colab.research.google.com/github/pluplu68/agent-lab/blob/main/RAG/RAG_Retrieval_Lab_v2_RealData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Retrieval Lab v2 — Real Data + HyDE + CRAG-inspired Routing

**A concise, GitHub-ready retrieval experiment using real research abstracts and relevance labels.**

| Pipeline | What it demonstrates | API key? |
|---|---|---|
| **BM25 → Dense → RRF → Reranker** | Real document ranking | **No** |
| **Recall@5 / MRR@5 / nDCG@10** | Evaluation against genuine relevance judgments | **No** |
| **CRAG-inspired correction** | Grade retrieved evidence → broaden retrieval or abstain | **No** |
| **HyDE** | LLM creates *hypothetical* passage → retrieve real documents | **Optional Groq key** |
| **Grounded answer** | LLM answers using real retrieved abstracts with document IDs | **Optional Groq key** |

**Data:** [BEIR SciFact](https://huggingface.co/datasets/Hyukkyu/beir-scifact), a public scientific information-retrieval dataset with **5,183 abstracts**, test queries, and relevance labels. We load it at runtime; we do **not** redistribute its data. The dataset is listed as **CC BY-NC 4.0** (review the terms before commercial reuse).

**Runtime:** Google Colab or Jupyter. CPU works; a GPU may accelerate local embedding/reranker inference. Internet is needed once to download **public** data and pretrained models. The main retrieval lab needs **no paid API and no authentication**.

**Important:** Our `CRAG-inspired` branch implements the *principle* of evidence grading and corrective retrieval, not the original CRAG trained evaluator, web-search extension, or decomposition/recomposition. HyDE's invented document is a **search aid, never a factual source**.

## 0 · Setup

Run the install cell once in Colab. Restart the kernel if a package upgrade asks you to. No LangChain or vector-database service is necessary: this demo uses **exact dense search over a NumPy matrix**, plus BM25 and a cross-encoder.

In [2]:
%pip -q install "datasets>=3.0" "rank-bm25>=0.2.2" "sentence-transformers>=3.0" "numpy>=1.24" "pandas>=2.0"

In [3]:
import math
import re
import time
import random
from functools import lru_cache
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## 1 · Load real documents, test queries and ground truth

- `corpus`: scientific paper **titles + abstracts**. One abstract = one retrieval unit here; this is **not** a full-paper chunking experiment.
- `queries`: real science claims/questions from SciFact.
- `qrels`: labeled relevant document IDs; used **only for evaluation**, never as a retrieval feature.

For reproducibility, we select a fixed random subset of test queries but **search the full corpus**. This avoids inflating retrieval performance by retaining only documents known to be relevant.

In [4]:
DATASET = "Hyukkyu/beir-scifact"
corpus_ds = load_dataset(DATASET, "corpus", split="test")
queries_ds = load_dataset(DATASET, "queries", split="test")
qrels_ds = load_dataset(DATASET, "qrels", split="test")

DOCS = [
    {
        "id": str(row["id"]),
        "title": str(row["title"] or ""),
        "text": str(row["text"] or ""),
    }
    for row in corpus_ds
]
for doc in DOCS:
    doc["content"] = (doc["title"] + ". " + doc["text"]).strip(". ")

DOC_BY_ID = {d["id"]: d for d in DOCS}
DOC_IDS = [d["id"] for d in DOCS]
DOC_TEXTS = [d["content"] for d in DOCS]
QUERY_BY_ID = {str(row["id"]): str(row["text"]) for row in queries_ds}

QRELS = defaultdict(dict)
for row in qrels_ds:
    qid, did = str(row["query-id"]), str(row["corpus-id"])
    if int(row["score"]) > 0:
        QRELS[qid][did] = int(row["score"])

eligible = sorted(qid for qid in QRELS if qid in QUERY_BY_ID)
EVAL_N = 12             # Raise to 30+ for a more stable comparison (takes longer).
EVAL_IDS = random.Random(SEED).sample(eligible, min(EVAL_N, len(eligible)))
EVAL_QUERIES = [(qid, QUERY_BY_ID[qid]) for qid in EVAL_IDS]

assert len(DOC_BY_ID) == len(DOCS), "Document IDs must be unique."
assert all(set(QRELS[qid]) <= set(DOC_BY_ID) for qid in EVAL_IDS)
print(f"Corpus: {len(DOCS):,} real abstracts | Test queries with labels: {len(eligible)}")
print("One query:", EVAL_QUERIES[0][1])
print("One abstract:", DOCS[0]["content"][:260], "...")

README.md:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

corpus/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.09MB            

corpus/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

queries/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.8kB            

queries/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (7 outputs left out)

qrels/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Corpus: 5,183 real abstracts | Test queries with labels: 300
One query: The risk of breast cancer among parous women increases with placental weight of pregnancies, and this association is strongest for premenopausal breast cancer.
One abstract: Structural Homeostasis: Compensatory Adjustments of Dendritic Arbor Geometry in Response to Variations of Synaptic Input . As the nervous system develops, there is an inherent variability in the connections formed between differentiating neurons. Despite this  ...


## 2 · BM25 vs Dense Retrieval

**BM25:** lexical term matching (good for exact technical terms). **Dense Retrieval:** one embedding per abstract (good for semantic similarity). The documents are embedded once; query embeddings are computed on demand. The dense step is exact matrix similarity, **not** an ANN index.

In [5]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")

def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())

BM25 = BM25Okapi([tokenize(doc) for doc in DOC_TEXTS])
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dimensional; English
encoder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
DOC_VECS = np.asarray(
    encoder.encode(
        DOC_TEXTS,
        batch_size=64 if DEVICE == "cuda" else 32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ), dtype=np.float32
)
print("Document embeddings:", DOC_VECS.shape)

@lru_cache(maxsize=512)
def query_vector(query: str) -> np.ndarray:
    return np.asarray(
        encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0],
        dtype=np.float32,
    )


def top_hits(scores: np.ndarray, top_k: int = 10) -> list[dict]:
    """Return ranked hits in a common, inspectable format."""
    order = np.argsort(-np.asarray(scores))[:top_k]
    return [{"id": DOC_IDS[i], "score": float(scores[i])} for i in order]


def bm25_search(query: str, top_k: int = 10) -> list[dict]:
    return top_hits(BM25.get_scores(tokenize(query)), top_k)


def dense_search(query: str, top_k: int = 10) -> list[dict]:
    return top_hits(DOC_VECS @ query_vector(query), top_k)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Document embeddings: (5183, 384)


In [6]:
query = EVAL_QUERIES[0][1]
print("QUERY:", query)
print("BM25:", bm25_search(query, 3))
print("DENSE:", dense_search(query, 3))

# Inspect actual retrieved documents, not just IDs.
def show_hits(hits: list[dict], max_chars: int = 210) -> pd.DataFrame:
    return pd.DataFrame([
        {"doc_id": h["id"], "score": round(h["score"], 3),
         "title": DOC_BY_ID[h["id"]]["title"],
         "abstract": DOC_BY_ID[h["id"]]["text"][:max_chars]}
        for h in hits
    ])

display(show_hits(dense_search(query, 3)))

QUERY: The risk of breast cancer among parous women increases with placental weight of pregnancies, and this association is strongest for premenopausal breast cancer.
BM25: [{'id': '13831842', 'score': 77.06212624205628}, {'id': '13765757', 'score': 76.34153954267647}, {'id': '37480103', 'score': 73.64643952484509}]
DENSE: [{'id': '17450673', 'score': 0.708377480506897}, {'id': '23557241', 'score': 0.707018256187439}, {'id': '27123743', 'score': 0.6840260624885559}]


,doc_id,score,title,abstract
0,17450673,0.708,Intrauterine environments and breast cancer ri...,"INTRODUCTION Various perinatal factors, includ..."
1,23557241,0.707,Intrauterine factors and risk of breast cancer...,BACKGROUND Emerging evidence suggests an assoc...
2,27123743,0.684,Role of birthweight in the etiology of breast ...,Breast cancer may originate in utero. We revie...


## 3 · Hybrid Search: RRF + Cross-Encoder Reranking

**RRF:** merges rank positions instead of combining incompatible BM25/cosine scores. **Cross-encoder:** reads each `(query, candidate)` pair jointly and reorders only a small candidate set (more expensive than bi-encoder retrieval).

\(\mathrm{RRF}(d)=\sum_j \frac{1}{60+\mathrm{rank}_j(d)}\)

In [28]:
def rrf_fuse(rankings: list[list[dict]], top_k: int = 10, k: int = 60) -> list[dict]:
    merged = defaultdict(float)
    for hits in rankings:
        for rank, hit in enumerate(hits, start=1):
            merged[hit["id"]] += 1.0 / (k + rank)
    return [
        {"id": did, "score": score}
        for did, score in sorted(merged.items(), key=lambda x: x[1], reverse=True)[:top_k]
    ]


def hybrid_search(query: str, top_k: int = 10, candidate_k: int = 30) -> list[dict]:
    return rrf_fuse([bm25_search(query, candidate_k), dense_search(query, candidate_k)], top_k)

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # English MS MARCO reranker
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)


def rerank(query: str, hits: list[dict], top_k: int = 5) -> list[dict]:
    if not hits:
        return []
    pairs = [(query, DOC_BY_ID[h["id"]]["content"]) for h in hits]
    raw_scores = np.asarray(reranker.predict(pairs, show_progress_bar=False), dtype=float).reshape(-1)
    ranked = [{"id": h["id"], "score": float(s)} for h, s in zip(hits, raw_scores)]
    return sorted(ranked, key=lambda h: h["score"], reverse=True)[:top_k]


def hybrid_rerank(query: str, top_k: int = 5, candidate_k: int = 30) -> list[dict]:
    return rerank(query, hybrid_search(query, top_k=candidate_k, candidate_k=candidate_k), top_k)

print("HYBRID:", [h["id"] for h in hybrid_search(query, 5)])
print("RERANK:", [h["id"] for h in hybrid_rerank(query, 5)])
display(show_hits(hybrid_rerank(query, 3)))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

HYBRID: ['3973445', '49556906', '1590744', '9752604', '24721866']
RERANK: ['3973445', '23974474', '1590744', '24721866', '7005276']


,doc_id,score,title,abstract
0,3973445,6.489,Phosphorylation of Janus kinase 1 (JAK1) by AM...,Adenosine 5′-monophosphate–activated protein k...
1,23974474,4.063,PKD1 Inhibits AMPKα2 through Phosphorylation o...,AMP-activated protein kinase (AMPK) is an ener...
2,1590744,3.455,AMP-Activated Protein Kinase: An Ubiquitous Si...,The AMP-activated protein kinase (AMPK) is a k...


## 4 · Evaluate on real relevance labels

For each test query, compare rankings against the provided **SciFact qrels**. Metrics:

- **Recall@5** = relevant documents retrieved in top 5 / all judged relevant documents.
- **MRR@5** = reciprocal rank of the first relevant document (0 if absent).
- **nDCG@10** = ranking quality with a logarithmic discount; higher-ranked relevant documents count more.

These are **retrieval metrics**, not generated-answer correctness. Small evaluation samples are instructional, not publication-quality benchmark scores. Different retrieval methods use the same queries and qrels.

In [29]:
def retrieval_metrics(hits: list[dict], relevance: dict[str, int]) -> dict[str, float]:
    ordered_ids = [h["id"] for h in hits]
    relevant_ids = set(relevance)
    recall5 = len(set(ordered_ids[:5]) & relevant_ids) / max(1, len(relevant_ids))
    first_rank = next((r for r, did in enumerate(ordered_ids[:5], 1) if did in relevant_ids), None)
    mrr5 = 1.0 / first_rank if first_rank else 0.0

    def dcg(rels):
        return sum((2 ** int(rel) - 1) / math.log2(r + 1) for r, rel in enumerate(rels, 1))

    dcg10 = dcg([relevance.get(did, 0) for did in ordered_ids[:10]])
    ideal10 = dcg(sorted(relevance.values(), reverse=True)[:10])
    ndcg10 = dcg10 / ideal10 if ideal10 else 0.0
    return {"Recall@5": recall5, "MRR@5": mrr5, "nDCG@10": ndcg10}

METHODS = {
    "BM25": lambda q: bm25_search(q, 10),
    "Dense": lambda q: dense_search(q, 10),
    "Hybrid RRF": lambda q: hybrid_search(q, 10),
    "Hybrid + Reranker": lambda q: hybrid_rerank(q, 10),
}

rows = []
for name, search in METHODS.items():
    observed = []
    started = time.perf_counter()
    for qid, query in EVAL_QUERIES:
        observed.append(retrieval_metrics(search(query), QRELS[qid]))
    seconds_per_query = (time.perf_counter() - started) / len(EVAL_QUERIES)
    average = pd.DataFrame(observed).mean().to_dict()
    rows.append({"method": name, **average, "seconds/query": seconds_per_query})

results_df = pd.DataFrame(rows).set_index("method")
display(results_df.round(3))
print("Note: timings include Python overhead and may reflect model warm-up/caching; not a production latency benchmark.")

,Recall@5,MRR@5,nDCG@10,seconds/query
method,,,,
BM25,0.757,0.595,0.654,0.018
Dense,0.787,0.629,0.658,0.001
Hybrid RRF,0.803,0.684,0.719,0.024
Hybrid + Reranker,0.703,0.601,0.649,0.286


Note: timings include Python overhead and may reflect model warm-up/caching; not a production latency benchmark.


## 5 · CRAG-inspired Retrieval Correction (API-free)

**Intuition:** retrieve → grade the evidence → if confidence is low, try a broader **hybrid retrieval**; if still weak, **abstain** rather than hallucinate.

Here the **cross-encoder** acts as a heuristic retrieval evaluator. We map its *raw logit* through a sigmoid to obtain a bounded **routing score, not a calibrated probability**. The illustrative threshold below should be tuned on a **separate validation set**, not on the final test qrels. This is a simplified CRAG-inspired pipeline; the original CRAG also includes web search and finer knowledge refinement.

In [9]:
def retrieval_grade(query: str, hits: list[dict]) -> float:
    """Heuristic relevance score, NOT calibrated statistical confidence."""
    if not hits:
        return 0.0
    pairs = [(query, DOC_BY_ID[h["id"]]["content"]) for h in hits]
    raw = np.asarray(reranker.predict(pairs, show_progress_bar=False), dtype=float).reshape(-1)
    max_logit = float(np.max(raw))
    return 1.0 / (1.0 + math.exp(-max(-40.0, min(40.0, max_logit))))


def crag_inspired(query: str, threshold: float = 0.65, top_k: int = 5) -> dict:
    """Dense retrieval → grade → broaden to hybrid + rerank → abstain if weak."""
    initial = dense_search(query, top_k=8)
    initial_grade = retrieval_grade(query, initial)
    if initial_grade >= threshold:
        return {"route": "USE_INITIAL", "score": initial_grade,
                "hits": rerank(query, initial, top_k)}

    candidates = hybrid_search(query, top_k=40, candidate_k=60)
    corrected = rerank(query, candidates, top_k=top_k)
    corrected_grade = retrieval_grade(query, corrected)
    if corrected_grade >= threshold:
        return {"route": "CORRECTED_HYBRID", "score": corrected_grade,
                "hits": corrected}
    return {"route": "ABSTAIN_LOW_EVIDENCE", "score": corrected_grade, "hits": []}

for question in [
    EVAL_QUERIES[0][1],
    "Who won the 2025 World Series?",  # outside this scientific abstract collection
]:
    outcome = crag_inspired(question)
    print("\nQUESTION:", question)
    print("Route:", outcome["route"], "| heuristic score:", round(outcome["score"], 3))
    if outcome["hits"]:
        display(show_hits(outcome["hits"][:2], max_chars=100))

# Do not interpret this route as a verified answer or a benchmark result.


QUESTION: The risk of breast cancer among parous women increases with placental weight of pregnancies, and this association is strongest for premenopausal breast cancer.
Route: USE_INITIAL | heuristic score: 0.993


,doc_id,score,title,abstract
0,13831842,4.983,Pooled analysis of prospective cohort studies ...,The association between anthropometric indices...
1,37480103,4.609,Pregnancy characteristics and maternal risk of...,"CONTEXT During pregnancy, serum levels of estr..."



QUESTION: Who won the 2025 World Series?
Route: ABSTAIN_LOW_EVIDENCE | heuristic score: 0.0


## 6 · HyDE + evidence-grounded generation

### Why an API here?

**HyDE:** Have an LLM write a *hypothetical* passage matching the user's question, then use the **passage embedding** to retrieve real documents. The imaginary passage is **not** treated as evidence. This may help or harm retrieval: compare it with the ordinary dense baseline.

**Optional answer generation:** Send only the **public SciFact abstracts retrieved by the model** to an LLM and ask for an evidence-grounded answer with doc IDs. This is RAG, not a retrieval metric.

We use [Groq](https://console.groq.com/keys) as one free-tier API option ([official docs](https://console.groq.com/docs/quickstart), [current supported models](https://console.groq.com/docs/models)). **Free-tier quotas, availability and model IDs can change.** The main notebook works without it.

**Security:** Never put your API key in a GitHub notebook or print it. In Colab, store `GROQ_API_KEY` in **Secrets**, or enter it when prompted with `getpass`. Use public documents only; sending sensitive data to external inference services requires approval and a privacy review.

In [30]:
# This cell is optional. It installs the SDK but makes NO API call.
RUN_API_DEMOS = True  # Change to True only after getting a Groq API key.
GROQ_MODEL = "openai/gpt-oss-20b"  # Officially supported as of Nov 2026; check Groq models page.

if RUN_API_DEMOS:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "groq>=0.28"])

In [17]:
import os
from getpass import getpass


def make_groq_client():
    from groq import Groq
    key = os.environ.get("GROQ_API_KEY", "")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GROQ_API_KEY") or ""
        except Exception:  # No Colab environment or secret not configured.
            pass
    if not key:
        key = getpass("Enter Groq API key (hidden; not saved to notebook): ")
    if not key:
        raise ValueError("No GROQ_API_KEY given. Leave RUN_API_DEMOS=False to skip.")
    return Groq(api_key=key)



def groq_text(client, prompt: str, max_tokens: int = 300) -> str:
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=max(1024, max_tokens * 4),
        reasoning_effort="low",
        temperature=0.1,
    )
    text = (response.choices[0].message.content or "").strip()
    if not text:
        print("Finish reason:", response.choices[0].finish_reason)
        print("Token usage:", response.usage)
        raise RuntimeError("Groq returned empty content.")
    return text

def hyde_retrieve(query: str, client, top_k: int = 5):
    hypothetical = groq_text(
        client,
        "Write an approximately 90-word *hypothetical* scientific abstract passage "
        "that could be relevant to the following scientific question or claim. "
        "This is for retrieval only, not a verified factual answer. "
        f"Question/claim: {query}",
        max_tokens=230,
    )
    v = np.asarray(encoder.encode([hypothetical], normalize_embeddings=True, convert_to_numpy=True)[0])
    return hypothetical, top_hits(DOC_VECS @ v, top_k)


def grounded_answer(query: str, hits: list[dict], client) -> str:
    evidence = "\n\n".join(
        f'[Document {h["id"]}] {DOC_BY_ID[h["id"]]["content"][:1300]}'
        for h in hits[:3]
    )
    prompt = (
        "Answer the question using ONLY the evidence snippets. "
        "Cite supporting document IDs exactly as [Document ID]. "
        "If evidence is insufficient, say so; do not invent studies or citations.\n\n"
        f"QUESTION: {query}\n\nEVIDENCE:\n{evidence}\n\nANSWER:"
    )
    return groq_text(client, prompt, max_tokens=400)

In [18]:
if RUN_API_DEMOS:
    client = make_groq_client()
    question = EVAL_QUERIES[0][1]
    ordinary = dense_search(question, top_k=5)
    hypothetical, hyde_hits = hyde_retrieve(question, client, top_k=5)

    print("QUESTION:", question)
    print("\nHYPOTHETICAL PASSAGE (NOT EVIDENCE):\n", hypothetical[:700])
    print("\nDense IDs:", [h["id"] for h in ordinary])
    print("HyDE IDs: ", [h["id"] for h in hyde_hits])
    qid = EVAL_QUERIES[0][0]
    print("Dense retrieval:", retrieval_metrics(ordinary, QRELS[qid]))
    print("HyDE retrieval: ", retrieval_metrics(hyde_hits, QRELS[qid]))

    print("\nGROUNDED ANSWER (from retrieved REAL docs, not imaginary passage):")
    print(grounded_answer(question, hybrid_rerank(question, top_k=3), client))
else:
    print("Optional API demos skipped. Core retrieval, evaluation, and CRAG-inspired code still run.")

QUESTION: The risk of breast cancer among parous women increases with placental weight of pregnancies, and this association is strongest for premenopausal breast cancer.

HYPOTHETICAL PASSAGE (NOT EVIDENCE):
 **Abstract (hypothetical)**  
We investigated the relationship between placental weight and subsequent breast cancer risk in a prospective cohort of 12,345 parous women followed for 25 years. Placental weight was recorded at delivery and categorized into quartiles. Cox proportional hazards models adjusted for age, parity, BMI, and family history revealed a dose‑response association: women in the highest quartile (> 450 g) had a 1.45‑fold increased risk of premenopausal breast cancer (HR = 1.45; 95 % CI 1.20–1.75) compared with the lowest quartile (< 300 g). The association attenuated for postmenopausal disease (HR = 1.12; 95 % CI 0.95–1.32). These findings suggest that placental mass, reflec

Dense IDs: ['17450673', '23557241', '27123743', '16322674', '37480103']
HyDE IDs:  ['2355

# more test

In [35]:
# Select 30 reproducible queries with relevance labels
#EVAL_N = 30
EVAL_N = 100

EVAL_IDS = random.Random(SEED).sample(
    eligible, min(EVAL_N, len(eligible))
)
EVAL_QUERIES = [(qid, QUERY_BY_ID[qid]) for qid in EVAL_IDS]

print(f"Selected {len(EVAL_QUERIES)} queries")

Selected 100 queries


In [36]:
# Unified benchmark: 5 methods, same 30 SciFact queries

# Reuse the existing Groq client, or create one
if "client" not in globals():
    client = make_groq_client()

METHODS = {
    "BM25": lambda q: bm25_search(q, top_k=10),
    "Dense": lambda q: dense_search(q, top_k=10),
    "Hybrid RRF": lambda q: hybrid_search(q, top_k=10),
    "Hybrid + Reranker": lambda q: hybrid_rerank(q, top_k=10),
}

# Cache HyDE outputs to avoid repeated API charges/calls
HYDE_CACHE = globals().get("HYDE_CACHE", {})
rows = []

for i, (qid, question) in enumerate(EVAL_QUERIES, 1):

    # 1. Compare four API-free retrieval methods
    for method, search_fn in METHODS.items():
        start = time.perf_counter()
        hits = search_fn(question)
        latency = time.perf_counter() - start

        metrics = retrieval_metrics(hits, QRELS[qid])

        rows.append({
            "query_id": qid,
            "method": method,
            **metrics,
            "seconds/query": latency
        })

    # 2. HyDE: Groq API + hypothetical embedding
    if qid not in HYDE_CACHE:
        start = time.perf_counter()
        hypothetical, hyde_hits = hyde_retrieve(
            question, client, top_k=10
        )
        latency = time.perf_counter() - start
        HYDE_CACHE[qid] = (hyde_hits, latency)

        # Reduce risk of exceeding free API rate limits
        time.sleep(3)

    hyde_hits, latency = HYDE_CACHE[qid]
    metrics = retrieval_metrics(hyde_hits, QRELS[qid])

    rows.append({
        "query_id": qid,
        "method": "HyDE",
        **metrics,
        "seconds/query": latency
    })

    if i % 5 == 0:
        print(f"Completed {i}/{len(EVAL_QUERIES)} queries")

# 3. Aggregate results
benchmark_df = pd.DataFrame(rows)

summary = (
    benchmark_df
    .groupby("method")[
        ["Recall@5", "MRR@5", "nDCG@10", "seconds/query"]
    ]
    .mean()
    .sort_values("nDCG@10", ascending=False)
    .round(3)
)

display(summary)

Completed 5/100 queries
Completed 10/100 queries
Completed 15/100 queries
Completed 20/100 queries
Completed 25/100 queries
Completed 30/100 queries
Completed 35/100 queries
Completed 40/100 queries
Completed 45/100 queries
Completed 50/100 queries
Completed 55/100 queries
Completed 60/100 queries
Completed 65/100 queries
Completed 70/100 queries
Completed 75/100 queries
Completed 80/100 queries
Completed 85/100 queries
Completed 90/100 queries
Completed 95/100 queries
Completed 100/100 queries


,Recall@5,MRR@5,nDCG@10,seconds/query
method,,,,
Hybrid RRF,0.805,0.719,0.758,0.036
Hybrid + Reranker,0.779,0.688,0.721,0.339
Dense,0.813,0.669,0.706,0.008
HyDE,0.786,0.653,0.704,0.438
BM25,0.773,0.651,0.692,0.032


In [33]:
benchmark_df.to_csv("rag_benchmark_30_queries.csv", index=False)
summary.to_csv("rag_benchmark_summary.csv")

display(summary)

,Recall@5,MRR@5,nDCG@10,seconds/query
method,,,,
Hybrid RRF,0.803,0.684,0.719,0.051
HyDE,0.747,0.637,0.688,0.464
Dense,0.787,0.629,0.658,0.003
BM25,0.757,0.595,0.654,0.050
Hybrid + Reranker,0.703,0.601,0.649,0.348


In [26]:
comparison = benchmark_df.pivot(
    index="query_id",
    columns="method",
    values="nDCG@10"
)

comparison["rerank_delta"] = (
    comparison["Hybrid + Reranker"]
    - comparison["Hybrid RRF"]
)

# Find queries where reranking hurts the most
display(
    comparison.sort_values("rerank_delta").head(10)
)

method,BM25,Dense,HyDE,Hybrid + Reranker,Hybrid RRF,rerank_delta
query_id,,,,,,
324,0.430677,0.630930,0.315465,0.000000,1.000000,-1.000000
1204,1.000000,1.000000,1.000000,0.500000,1.000000,-0.500000
185,0.630930,1.000000,1.000000,0.500000,1.000000,-0.500000
213,0.356207,0.386853,0.430677,0.000000,0.386853,-0.386853
54,0.315465,1.000000,1.000000,0.333333,0.630930,-0.297596
133,0.131205,0.639945,0.554282,0.529635,0.797254,-0.267619
1099,0.500000,0.386853,0.500000,0.386853,0.630930,-0.244077
1100,0.630930,0.386853,0.315465,0.430677,0.630930,-0.200253
198,0.500000,0.000000,1.000000,0.333333,0.356207,-0.022874


In [27]:
# Inspect the largest reranking failure
qid, question = next(
    (qid, q) for qid, q in EVAL_QUERIES
    if str(qid) == "324"
)

print("QUERY:", question)
print("\nGROUND TRUTH:", QRELS[qid])

methods = {
    "Hybrid RRF": hybrid_search,
    "Hybrid + Reranker": hybrid_rerank,
}

for name, search_fn in methods.items():
    hits = search_fn(question, top_k=10)

    print(f"\n{'=' * 20} {name}")
    print("Metrics:", retrieval_metrics(hits, QRELS[qid]))

    for rank, hit in enumerate(hits, 1):
        print(f"{rank}. {hit}")

QUERY: Deleting Raptor reduces G-CSF levels.

GROUND TRUTH: {'2014909': 1}

==================== Hybrid RRF
Metrics: {'Recall@5': 1.0, 'MRR@5': 1.0, 'nDCG@10': 1.0}
1. {'id': '2014909', 'score': 0.031754032258064516}
2. {'id': '4658268', 'score': 0.0315136476426799}
3. {'id': '22174015', 'score': 0.02921395544346364}
4. {'id': '24612804', 'score': 0.02844163539403516}
5. {'id': '23816832', 'score': 0.026190476190476188}
6. {'id': '8724666', 'score': 0.015873015873015872}
7. {'id': '6962472', 'score': 0.015873015873015872}
8. {'id': '10790846', 'score': 0.015625}
9. {'id': '1818578', 'score': 0.015384615384615385}
10. {'id': '46565020', 'score': 0.015151515151515152}

==================== Hybrid + Reranker
Metrics: {'Recall@5': 0.0, 'MRR@5': 0.0, 'nDCG@10': 0.0}
1. {'id': '22174015', 'score': -0.08123563230037689}
2. {'id': '8724666', 'score': -1.3190608024597168}
3. {'id': '5260382', 'score': -1.7781044244766235}
4. {'id': '46565020', 'score': -3.2313597202301025}
5. {'id': '654735', '

# New experiment
使用完全相同的 30 /100 个 SciFact Queries。
使用完全相同的 30 /100 个候选文档。
都返回 Top-10。
唯一改变的是 Ranking Method。

In [37]:
# Experiment: General vs Biomedical Reranker
from sentence_transformers import CrossEncoder

# 1. Load biomedical reranker
medcpt = CrossEncoder(
    "ncbi/MedCPT-Cross-Encoder",
    max_length=512
)

# 2. Rerank the same candidate documents
def medcpt_rerank(query, candidates, top_k=10):
    pairs = [
        (query, DOC_BY_ID[str(h["id"])]["content"])
        for h in candidates
    ]

    scores = medcpt.predict(pairs, batch_size=16)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: float(x[1]),
        reverse=True
    )

    return [
        {"id": h["id"], "score": float(score)}
        for h, score in ranked[:top_k]
    ]

# 3. Evaluate all three methods on the same 30 queries
rows = []

for qid, question in EVAL_QUERIES:

    candidates = hybrid_search(
        question, top_k=30, candidate_k=30
    )

    methods = {
        "Hybrid RRF": candidates[:10],
        "MS MARCO Reranker": rerank(
            question, candidates, top_k=10
        ),
        "MedCPT Reranker": medcpt_rerank(
            question, candidates, top_k=10
        )
    }

    for name, hits in methods.items():
        rows.append({
            "query_id": qid,
            "method": name,
            **retrieval_metrics(hits, QRELS[qid])
        })

# 4. Compare results
reranker_df = pd.DataFrame(rows)

display(
    reranker_df.groupby("method")[
        ["Recall@5", "MRR@5", "nDCG@10"]
    ].mean().round(3)
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,Recall@5,MRR@5,nDCG@10
method,,,
Hybrid RRF,0.805,0.719,0.758
MS MARCO Reranker,0.779,0.688,0.721
MedCPT Reranker,0.868,0.789,0.817


## Paired Bootstrap 95% Confidence Interval
目的：检查 MedCPT 相比 Hybrid RRF 的 nDCG@10 提升是否稳定。

In [38]:
import numpy as np
import pandas as pd

# Match results by query ID
paired = reranker_df.pivot(
    index="query_id",
    columns="method",
    values="nDCG@10"
).dropna(subset=["Hybrid RRF", "MedCPT Reranker"])

# Per-query improvement
delta = (
    paired["MedCPT Reranker"] - paired["Hybrid RRF"]
).to_numpy()

# Paired bootstrap
rng = np.random.default_rng(42)
n_bootstrap = 5000

boot_means = np.array([
    rng.choice(delta, size=len(delta), replace=True).mean()
    for _ in range(n_bootstrap)
])

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

print(f"Queries: {len(delta)}")
print(f"Mean improvement: {delta.mean():.4f}")
print(f"95% Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Improved queries: {(delta > 0).sum()}")
print(f"Worse queries: {(delta < 0).sum()}")
print(f"Unchanged queries: {(delta == 0).sum()}")


Queries: 100
Mean improvement: 0.0593
95% Bootstrap CI: [0.0246, 0.0965]
Improved queries: 22
Worse queries: 6
Unchanged queries: 72


## Latency Benchmark
目的：比较三种方法的平均运行时间以及 P95 Latency（95% 的请求在该时间内完成）。

In [39]:
import time

# Warm up models
q = EVAL_QUERIES[0][1]
c = hybrid_search(q, top_k=30, candidate_k=30)
rerank(q, c, top_k=10)
medcpt_rerank(q, c, top_k=10)

latency_rows = []

for qid, question in EVAL_QUERIES:
    # Stage 1: Hybrid retrieval
    t0 = time.perf_counter()
    candidates = hybrid_search(
        question, top_k=30, candidate_k=30
    )
    t1 = time.perf_counter()

    # Stage 2: MS MARCO
    rerank(question, candidates, top_k=10)
    t2 = time.perf_counter()

    # Stage 3: MedCPT
    medcpt_rerank(question, candidates, top_k=10)
    t3 = time.perf_counter()

    retrieval_time = t1 - t0

    latency_rows.extend([
        {
            "method": "Hybrid RRF",
            "latency_ms": retrieval_time * 1000
        },
        {
            "method": "MS MARCO Reranker",
            "latency_ms": (retrieval_time + t2 - t1) * 1000
        },
        {
            "method": "MedCPT Reranker",
            "latency_ms": (retrieval_time + t3 - t2) * 1000
        }
    ])

latency_df = pd.DataFrame(latency_rows)

latency_summary = latency_df.groupby("method")["latency_ms"].agg(
    mean_ms="mean",
    p95_ms=lambda x: x.quantile(0.95)
).round(2)

display(latency_summary)


,mean_ms,p95_ms
method,,
Hybrid RRF,29.76,57.13
MS MARCO Reranker,282.52,353.44
MedCPT Reranker,916.46,1133.78


## 7 · Sanity checks and interpretation

**Please read before reporting results:**

- SciFact contains **real abstracts + real relevance judgments**. No retrieval scores are hardcoded; results depend on the downloaded models.
- **BM25 vs Dense vs RRF vs Reranker** can produce different rankings; none is guaranteed best on every query or dataset.
- CRAG's threshold is illustrative and **not calibrated**; never claim that `score=0.7` implies a 70% probability of correctness.
- **HyDE** incurs an external LLM request and can hallucinate; its output only forms a retrieval query, never evidence.
- This demo is *not* full-text PDF chunking, ANN indexing, a production vector database, or an end-to-end hallucination benchmark.
- For a stronger portfolio: expand the evaluation set, tune hyperparameters on validation data, record dependency versions/seeds, run on another BEIR corpus, and add tests/CI.

**Further reading:** [BEIR](https://github.com/beir-cellar/beir) · [HyDE](https://arxiv.org/abs/2212.10496) · [CRAG](https://arxiv.org/abs/2401.15884) · [Groq API](https://console.groq.com/docs/quickstart)

In [19]:
# Pure algorithm tests: do not depend on live API calls or model-specific scores.
assert tokenize("RAG: Token-Level retrieval!") == ["rag", "token", "level", "retrieval"]
assert [h["id"] for h in rrf_fuse(
    [[{"id":"a","score":1}, {"id":"b","score":0}],
     [{"id":"b","score":1}, {"id":"a","score":0}]], top_k=2
)] == ["a", "b"] or set(h["id"] for h in rrf_fuse(
    [[{"id":"a","score":1}, {"id":"b","score":0}],
     [{"id":"b","score":1}, {"id":"a","score":0}]], top_k=2
)) == {"a", "b"}  # tie: either order is acceptable
assert retrieval_metrics(
    [{"id":"a","score":1}, {"id":"b","score":0}], {"a":1}
)["MRR@5"] == 1.0
assert DOC_VECS.shape[0] == len(DOCS)
print("Sanity checks passed. All displayed retrieval data came from SciFact.")

Sanity checks passed. All displayed retrieval data came from SciFact.
